In [2]:
print("Hi")

Hi


In [3]:
%pwd

'd:\\Medical-Chatbot\\research'

In [4]:
import os
os.chdir("../")

In [5]:
%pwd

'd:\\Medical-Chatbot'

In [6]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [9]:
# Extract data from PDF files
def load_pdf_file(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [10]:
extracted_data = load_pdf_file(data="Data/")

In [16]:
# Split the data into text chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [17]:
text_chunks = text_split(extracted_data)

In [18]:
len(text_chunks)

7093

In [19]:
from langchain.embeddings import HuggingFaceEmbeddings

In [20]:
# 384 dimensional vector embeddings
def download_huggingface_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [21]:
embeddings = download_huggingface_embeddings()

C:\Users\atish\AppData\Local\Temp\ipykernel_12696\2912381467.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
d:\Medical-Chatbot\medibotVenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
query_result = embeddings.embed_query("Hello world")
print("length", len(query_result))

length 384


In [27]:
from dotenv import load_dotenv
load_dotenv()

True

In [36]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")


In [30]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medicalbot"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

In [37]:
import os
# Set the Pinecone API key as an environment variable so that we don't need to write it frequently
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

TypeError: str expected, not NoneType

In [32]:
# Embed each chunk and upsert embedding vectors to Pinecone
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name,
)

In [33]:
# Load existing index
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name=index_name,
)

In [34]:
docsearch

In [35]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriever.get_relevant_documents("What are the symptoms of diabetes?")

C:\Users\atish\AppData\Local\Temp\ipykernel_12696\1648589189.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  retriever.get_relevant_documents("What are the symptoms of diabetes?")


[Document(metadata={'page': 436.0, 'source': 'Data\\The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf'}, page_content='adrenergic agonists. Other medications that can cause\ndiabetes symptoms include isoniazid, nicotinic acid,\ncimetidine, and heparin.\nSymptoms\nSymptoms of diabetes can develop suddenly (over\ndays or weeks) in previously healthy children or adoles-\ncents, or can develop gradually (over several years) in\noverweight adults over the age of 40. The classic symp-\ntoms include feeling tired and sick, frequent urination,\nexcessive thirst, excessive hunger, and weight loss.'),
 Document(metadata={'page': 435.0, 'source': 'Data\\The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf'}, page_content='glucose in the blood cannot be absorbed into the cells of\nthe body. Symptoms include frequent urination, lethargy,\nexcessive thirst, and hunger. The treatment includes\nchanges in diet, oral medications, and in some cases,\ndaily injections of insulin.\nDescription\nDiabetes mellitus is a 

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key="your_api_key",
    model_name="llama3-8b-8192"
)
